# Clase 141 — Encoder-Decoder para traducción

La arquitectura **seq2seq** (Sutskever et al. 2014): un **encoder** comprime la
oración fuente en un estado de contexto y un **decoder** genera la traducción token
a token. Vemos **teacher forcing**, tokens `[start]`/`[end]`, y **BLEU**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `nltk` (para BLEU).

## 1. Pares fuente-destino y tokens especiales

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
pares = [
    ("hello", "hola"),
    ("good morning", "buenos dias"),
    ("thank you", "gracias"),
    ("how are you", "como estas"),
    ("see you later", "hasta luego"),
]
fuente = [s for s, _ in pares]
destino = ["[start] " + t + " [end]" for _, t in pares]   # marcadores de inicio/fin
print(destino[0])

## 2. Vectorizadores independientes para fuente y destino

In [ ]:
VOCAB, LEN = 1000, 8
vec_src = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN)
vec_tgt = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN + 1)
vec_src.adapt(fuente)
vec_tgt.adapt(destino)
print("vocab destino:", len(vec_tgt.get_vocabulary()))

## 3. Encoder: `LSTM(return_state=True)` → estado de contexto

In [ ]:
DIM = 256
enc_in = keras.Input(shape=(1,), dtype="string")
ex = vec_src(enc_in)
ex = layers.Embedding(VOCAB, DIM, mask_zero=True)(ex)
_, state_h, state_c = layers.LSTM(DIM, return_state=True)(ex)   # (h_T, c_T) = contexto
encoder = keras.Model(enc_in, [state_h, state_c], name="encoder")
encoder.summary()

## 4. Decoder con teacher forcing (`initial_state` = estado del encoder)

In [ ]:
dec_in = keras.Input(shape=(1,), dtype="string")
dx = vec_tgt(dec_in)
dx = layers.Embedding(VOCAB, DIM, mask_zero=True)(dx)
dx = layers.LSTM(DIM, return_sequences=True)(dx, initial_state=[state_h, state_c])
salida = layers.Dense(VOCAB, activation="softmax")(dx)
seq2seq = keras.Model([enc_in, dec_in], salida, name="seq2seq")
seq2seq.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
# Teacher forcing: en training el decoder ve el target real (shifted), no su predicción.
seq2seq.summary()

## 5. Inference autoregresiva (bucle `[start]` → … → `[end]`)

In [ ]:
vocab_tgt = vec_tgt.get_vocabulary()
def traducir(texto, max_len=LEN):
    h, c = encoder.predict(np.array([texto]), verbose=0)   # contexto de la fuente
    generado = "[start]"
    gen = np.random.default_rng(0)
    for _ in range(max_len):
        # Ilustrativo: en un decoder real se reusa el estado y se realimenta la predicción.
        siguiente = vocab_tgt[gen.integers(2, len(vocab_tgt))]
        if siguiente == "[end]":
            break
        generado += " " + siguiente
    return generado.replace("[start]", "").strip()

print("traducción (modelo sin entrenar, ilustrativo):", traducir("hello"))

## 6. Evaluar con BLEU

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
suave = SmoothingFunction().method1
ref = [["hola", "buenos", "dias"]]
print("BLEU perfecto:", round(sentence_bleu(ref, ["hola", "buenos", "dias"], smoothing_function=suave), 3))
print("BLEU parcial :", round(sentence_bleu(ref, ["hola", "tardes"], smoothing_function=suave), 3))

## Ejercicios

1. **Preparar datos**: tokenizá fuente y destino, agregá `[start]`/`[end]` y padding.
2. **Encoder**: `Embedding → LSTM(256, return_state=True)`, conservá `state_h`, `state_c`.
3. **Decoder en training**: teacher forcing con `initial_state = encoder_state`.
4. **BLEU**: calculá BLEU sobre un test set con `nltk` y `SmoothingFunction`.

## Conclusiones

- **seq2seq**: encoder resume la fuente en `(h_T, c_T)`; decoder genera desde ese estado.
- **Teacher forcing** acelera el training (target real como input) pero crea mismatch con inference.
- Los tokens `[start]`/`[end]` marcan el inicio y el fin de la generación.
- El **cuello de botella** (todo el significado en un vector fijo) motivó la **atención** (clase 142).
- **BLEU** mide solapamiento de n-gramas (0-100); es aceptable pero falla con paráfrasis.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — Preparar datos: tokens `[start]`/`[end]` + padding

El destino lleva marcadores de inicio/fin; vectorizadores independientes para fuente y destino, con padding a longitud fija.

In [ ]:
from tensorflow.keras import layers

pares = [("hello", "hola"), ("good morning", "buenos dias"),
         ("thank you", "gracias"), ("how are you", "como estas")]
fuente = [s for s, _ in pares]
destino = ["[start] " + t + " [end]" for _, t in pares]

VOCAB, LEN = 1000, 8
vec_src = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN)
vec_tgt = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN + 1)
vec_src.adapt(fuente); vec_tgt.adapt(destino)
print("destino[0]:", destino[0], "| vocab destino:", len(vec_tgt.get_vocabulary()))

### Ejercicio 2 — Encoder: `LSTM(return_state=True)`

El encoder comprime la fuente en el par de estados `(state_h, state_c)`, el vector de contexto que inicializa el decoder.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

DIM = 256
enc_in = keras.Input(shape=(1,), dtype="string")
ex = vec_src(enc_in)
ex = layers.Embedding(VOCAB, DIM, mask_zero=True)(ex)
_, state_h, state_c = layers.LSTM(DIM, return_state=True)(ex)   # contexto (h_T, c_T)
encoder = keras.Model(enc_in, [state_h, state_c], name="encoder")
print("estados del encoder:", state_h.shape, state_c.shape)

### Ejercicio 3 — Decoder en training con teacher forcing

Se pasa el target real (shifted) como input y se arranca el LSTM del decoder con `initial_state = encoder_state`.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

dec_in = keras.Input(shape=(1,), dtype="string")
dx = vec_tgt(dec_in)
dx = layers.Embedding(VOCAB, DIM, mask_zero=True)(dx)
dx = layers.LSTM(DIM, return_sequences=True)(dx, initial_state=[state_h, state_c])
salida = layers.Dense(VOCAB, activation="softmax")(dx)
seq2seq = keras.Model([enc_in, dec_in], salida, name="seq2seq")
seq2seq.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
print("teacher forcing: el decoder ve el target real, arranca con el estado del encoder")

### Ejercicio 4 — Bucle de inferencia seq2seq CORRECTO (greedy decode con estados)

En inference no hay target: se genera token a token. La clave es **reusar los estados** `(h, c)` del decoder paso a paso y realimentar el `argmax`.

**Nota:** el cuerpo real necesita los modelos entrenados en GPU (aquí van comentados); lo que importa —la lógica de decodificación— queda explícita y ejecutable como estructura.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# --- modelos de inferencia derivados del entrenamiento ---
# Encoder: texto fuente -> [h, c]
#   encoder_inf = keras.Model(enc_in, [state_h, state_c])
# Decoder paso-a-paso: (token, h, c) -> (logits, h_new, c_new)
#   tok = keras.Input(shape=(1,), dtype="int32")
#   hin = keras.Input(shape=(DIM,)); cin = keras.Input(shape=(DIM,))
#   e = layers.Embedding(VOCAB, DIM)(tok)
#   seq, h2, c2 = layers.LSTM(DIM, return_sequences=True, return_state=True)(e, initial_state=[hin, cin])
#   logits = layers.Dense(VOCAB, activation="softmax")(seq)
#   decoder_step = keras.Model([tok, hin, cin], [logits, h2, c2])

def traducir_greedy(texto, encoder_inf, decoder_step, tgt_vocab, max_len=8):
    start_id, end_id = tgt_vocab.index("[start]"), tgt_vocab.index("[end]")
    h, c = encoder_inf.predict(np.array([texto]), verbose=0)   # 1) codifico la fuente
    token = np.array([[start_id]]); salida = []
    for _ in range(max_len):
        logits, h, c = decoder_step.predict([token, h, c], verbose=0)  # 2) reuso estado (h, c)
        next_id = int(np.argmax(logits[0, -1]))                        # 3) greedy: el más probable
        if next_id == end_id:                                          # 4) corto en [end]
            break
        salida.append(tgt_vocab[next_id])
        token = np.array([[next_id]])                                  # 5) realimento la predicción
    return " ".join(salida)

# print(traducir_greedy("hello", encoder_inf, decoder_step, vec_tgt.get_vocabulary()))
print("greedy decode: encode -> [start] -> argmax reusando (h,c) -> realimentar -> [end]")

### Ejercicio 5 — BLEU con `nltk` y `SmoothingFunction`

BLEU mide solapamiento de n-gramas (0-1); el *smoothing* evita ceros con oraciones cortas.

In [ ]:
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# suave = SmoothingFunction().method1
# ref = [["hola", "buenos", "dias"]]                 # referencia (lista de tokens)
# print("BLEU perfecto:", sentence_bleu(ref, ["hola", "buenos", "dias"], smoothing_function=suave))
# print("BLEU parcial :", sentence_bleu(ref, ["hola", "tardes"], smoothing_function=suave))
# corpus_bleu promedia sobre todo el test set.
print("BLEU: n-gram overlap con SmoothingFunction; 1.0 = traducción idéntica a la referencia")